# Défi du jour : Reranking avec Pinecone Serverless

## Pourquoi on fait ça ?
Les modèles de reranking améliorent la pertinence d'une recherche : ils attribuent un score de similarité entre une requête et des documents, puis réordonnent les résultats pour que les plus pertinents apparaissent en premier. Dans un contexte médical par exemple, ça permet à un clinicien d'accéder rapidement aux notes cliniques les plus critiques.

---

**Avant de te laisser copier-coller ce notebook, voici ce que je ne vais pas te laisser rater :**

1. **La fonction `get_embedding` de l'énoncé original a une erreur d'indentation Python.** Le corps de la fonction (à partir de `model_name = ...`) n'est pas indenté sous le `def`. Tel quel, ce code lève une `IndentationError` avant même d'atteindre le blanc à remplir. Je l'ai corrigé.
2. **Cette même fonction recharge le tokenizer et le modèle à chaque appel.** Si tu l'appelles plusieurs fois (une par question), tu retélécharges/recharges le modèle en mémoire à chaque fois — lent et inutile. Je l'ai sorti de la fonction.
3. **Le mean pooling "naïf" (`mean(dim=...)` sans masque d'attention) est techniquement faux** dès qu'il y a du padding : tu inclus des vecteurs de tokens `[PAD]` dans la moyenne, ce qui pollue l'embedding. Le modèle `all-MiniLM-L6-v2` est justement entraîné avec du *mean pooling pondéré par l'attention mask* (c'est documenté sur sa carte de modèle Hugging Face). Je te montre la version correcte, et je t'explique pourquoi la version "naïve" de l'énoncé n'est pas juste un détail cosmétique — surtout que le fichier JSONL fourni contient déjà des embeddings précalculés (probablement avec la vraie méthode). Si tes embeddings de requête ne sont pas calculés avec la même méthode que les embeddings indexés, tu compares des vecteurs qui ne vivent pas exactement dans le même espace, et ta recherche par similarité devient silencieusement moins fiable — sans qu'aucune erreur ne te le signale.
4. **`Authenticate()` de `pinecone_notebooks` ne fonctionne que dans Google Colab.** Si tu es sur Jupyter local, ça ne marchera pas — il faut définir `PINECONE_API_KEY` toi-même (variable d'environnement ou `os.environ["PINECONE_API_KEY"] = "ta_clé"`).
5. **Ne mets jamais ta clé API en dur dans un notebook que tu partages ou push sur GitHub.** L'énoncé insiste dessus, je le répète parce que c'est le genre d'erreur qui coûte cher.

Fait tourner ce notebook cellule par cellule, ne saute pas les explications — la partie "pourquoi" compte autant que le code.

## Partie 1 : Charger des documents et exécuter le modèle de reranking

In [ ]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

In [ ]:
import os

# Sur Colab : Authenticate() ouvre une fenêtre pour rentrer ta clé.
# En local (Jupyter classique) : définis la variable d'environnement toi-même.
if not os.environ.get("PINECONE_API_KEY"):
    try:
        from pinecone_notebooks.colab import Authenticate
        Authenticate()
    except ImportError:
        # On n'est pas dans Colab : on demande la clé manuellement
        os.environ["PINECONE_API_KEY"] = input("Colle ta clé API Pinecone ici : ")

In [ ]:
from pinecone import Pinecone

api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

In [ ]:
query = "Tell me about Apple's products"
documents = [
    "An apple a day keeps the doctor away, thanks to its fiber and vitamin C content.",         # fruit
    "Apple unveiled the new iPhone with an upgraded camera and faster A-series chip.",           # entreprise
    "Apple trees typically blossom in spring before producing fruit in early autumn.",           # fruit
    "Apple's MacBook lineup now ships with in-house Apple Silicon processors.",                  # entreprise
    "Cutting an apple into thin slices and adding cinnamon makes a simple, healthy snack."        # fruit
]

In [ ]:
from pinecone import RerankModel

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3
)

In [ ]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        print(f"{i+1}. score={m.score:.4f} | {m.document.text}")

# L'objet renvoyé par pc.inference.rerank() expose ses résultats via l'attribut .data
show_reranked_results(query, reranked.data)

**Ce que tu dois observer ici** : la requête parle de "produits Apple" (entreprise). Si le reranker fonctionne correctement, les documents sur l'iPhone/MacBook doivent remonter en tête, malgré la présence du même mot "apple" dans les documents sur le fruit. Si ce n'est pas le cas, ne te dis pas "c'est probablement normal" — creuse pourquoi (contexte trop court ? documents trop ambigus ?).

## Partie 2 : Configurer un index serverless pour des notes médicales

In [ ]:
!pip install pandas torch transformers

In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

spec = ServerlessSpec(cloud=cloud, region=region)

index_name = 'medical-notes-index'

In [ ]:
# Nettoyage d'un éventuel index préexistant du même nom
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

pc.create_index(
    name=index_name,
    dimension=384,   # doit correspondre à la sortie de all-MiniLM-L6-v2
    metric='cosine',
    spec=spec
)

## Partie 3 : Charger les données d'exemple

In [ ]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

In [ ]:
print("Data shape:", df.shape)
df.head()

**Vérifie toi-même** (l'énoncé ne te le demande pas explicitement, mais tu devrais le faire par réflexe avant tout upsert) : les colonnes contiennent bien `id`, `values` et `metadata` ? Si `values` fait autre chose que 384 nombres, ton `create_index(dimension=384, ...)` va planter à l'upsert, pas avant.

In [ ]:
# Contrôle de cohérence avant d'aller plus loin
assert set(["id", "values", "metadata"]).issubset(df.columns), "Colonnes manquantes dans le dataframe !"
assert len(df.iloc[0]["values"]) == 384, "La dimension des embeddings ne correspond pas à l'index créé !"
print("Structure du dataframe validée.")

## Partie 4 : Upserter les données dans l'index

In [ ]:
index = pc.Index(name=index_name)

index.upsert_from_dataframe(df)

In [ ]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

**Note critique** : `vector_count > 0` te dit juste "il y a au moins un vecteur", pas "tous les vecteurs attendus sont là". Si ton upsert échoue à mi-chemin (timeout réseau par exemple), cette boucle va quand même se terminer et te faire croire que tout est prêt. Une vraie vérification comparerait `vector_count` à `len(df)`.

## Partie 5 : Fonction d'embedding et requête

In [ ]:
# Corrections apportées par rapport à l'énoncé original :
# 1. Le tokenizer et le modèle sont chargés UNE SEULE FOIS, en dehors de la fonction
#    (l'énoncé les rechargeait à chaque appel : lent et inutile).
# 2. Le pooling utilise le masque d'attention (mean pooling correct), pas une simple
#    moyenne brute qui inclurait les tokens de padding.

model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
embed_model = AutoModel.from_pretrained(model_name)


def get_embedding(input_question):
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = embed_model(**encoded_input)

    token_embeddings = model_output.last_hidden_state  # (batch, seq_len, hidden)
    attention_mask = encoded_input['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()

    # Moyenne pondérée par le masque : on ignore les tokens de padding
    summed = torch.sum(token_embeddings * attention_mask, dim=1)
    counted = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
    embedding = summed / counted

    return embedding[0]

In [ ]:
question = "patient has chest pain"
query = get_embedding(question).tolist()

results = index.query(vector=[query], top_k=8, include_metadata=True)

sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

## Partie 6 : Afficher puis reranker les notes cliniques

In [ ]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}')
        print(f' Metadata: {match["metadata"]}')
        print('')

show_results(question, sorted_matches)

In [ ]:
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

In [ ]:
refined_query = "patient needs knee surgery"

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)

In [ ]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match.score}')
        print(f' Reranking Field: {match.document.reranking_field}')
        print('')

show_reranked_results(refined_query, reranked_results.data)

**Un point que l'exercice ne te fait jamais vérifier explicitement, mais qui compte** : compare `sorted_matches` (tri par score de similarité cosinus brut) et `reranked_results.data` (tri par score du reranker). Si l'ordre est identique dans les deux cas, ton reranking n'apporte rien sur cet exemple précis — et il faut te demander pourquoi (requête pas assez ambiguë ? champ `reranking_field` peu informatif ?) plutôt que de simplement cocher la case "j'ai fait tourner le reranker".

In [ ]:
# Nettoyage (à décommenter quand tu as fini, sinon l'index continue de tourner en facturation serverless)
# pc.delete_index(name=index_name)

## Bilan sans complaisance

- Ce notebook confirme surtout que le reranker sait distinguer "Apple entreprise" de "apple fruit" — un cas d'usage assez facile. Ça ne prouve rien sur des cas ambigus plus subtils (jargon médical proche, synonymes). Ne généralise pas la conclusion "le reranking marche bien" à partir de ce seul exemple.
- Le pipeline mélange deux sources d'embeddings différentes sans jamais le vérifier explicitement : ceux précalculés dans le JSONL (méthode inconnue) et ceux que tu calcules toi-même pour la requête (méthode que tu contrôles, ici corrigée). S'ils ne sont pas calculés de la même façon, la comparaison par similarité cosinus reste valide mathématiquement mais peut être moins précise sémantiquement. Si tu veux être rigoureux, regarde comment le JSONL a été généré avant de faire confiance aux résultats.
- Rien dans l'exercice ne te fait mesurer objectivement si le reranking améliore la pertinence (pas de ground truth, pas de métrique comme NDCG). Tu juges "à l'œil" en lisant le texte des notes — ce qui est correct pour un exercice pédagogique, mais ne confonds pas ça avec une évaluation rigoureuse en production.